# DL4CV PA02-Q5: Dropout


---

### $\color{red}{\textbf{Name and ID:}}$ XXXXXXXX YourName(Fill here)

Before we start, please put your student ID and your name in following format.
For example

12345678 HONGGildong


---

### Submission

1. The output of the code cells in this `.ipynb` file $\color{red}{\text{MUST}}$ be saved.
   - Save text output of code cells
   - Save graphs and plots of code cells    
2. All The `.py` files should be submitted.
3. zip the required `.ipynb` and `.py` files into $\color{red}{\mathtt{StudentID\_StudentName.zip}}$ and submit.
   - For example `12345678_HONGGildong.zip`
4. Deadline is $\color{red}{2026:10:17-23:59:59}$ (YYYY:MM:DD-HH:MM:SS)

---

### To-do

1. Fill in Your ID and Name.
2. (Programming) In `layers.py`, implement **dropout_forward()** function.
3. (Programming) In `layers.py`, implement **dropout_backward()** function.
6. (Programming) In `two_layer_net.py`, implement **`__init__()`** function in `class TwoLayerNet` with `dropout`.
7. (Programming) In `two_layer_net.py`, implement **loss()** function in `class TwoLayerNet` with `dropout`.
11. (Save Outputs) Successfully Run all the **code cells** in this `.ipynb`, $\color{red}{\text{MUST}}$ save the outputs.
12. Submit $\color{red}{\mathtt{StudentID\_StudentName.zip}}$ file to Plato before deadline.


---

## Dropout Layer Programming Assignment

In this programming assignment, you will implement the forward and backward passes of a Dropout layer and integrate Dropout into a two-layer neural network.

Dropout is a regularization technique used to reduce overfitting in neural networks. During training, Dropout randomly disables a fraction of neurons. As a result, the network cannot rely too strongly on any particular hidden unit and is encouraged to learn features that remain useful under different combinations of active neurons.

In this Programming Assignment you will:
    
- implement forward and backward pass of dropout layer.
- integrate Dropout into a two-layer neural network.

---

## Setup Code

### Google Colab Setup (Skip If Local Machine)

Unzip the `PA02.zip` file, upload entire `DeepLearning4ComputerVision` folder into Google drivers `My Drive` folder.

Next we need to run a few commands to set up our environment on Google Colab. If you are running this notebook on a local machine you can skip this section.

Run the following cell to mount your Google Drive. Follow the link, sign in to your Google account (the same account you used to store this notebook!) and copy the authorization code into the text box that appears below.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

GOOGLE_DRIVE_PATH_AFTER_MYDRIVE = 'DeepLearning4ComputerVision/PA02'

import os
import sys

#sys.path.append('/content/drive/My Drive/{}'.format(GOOGLE_DRIVE_PATH_AFTER_MYDRIVE)) # if not working, try this line instead
os.chdir('/content/drive/My Drive/{}'.format(GOOGLE_DRIVE_PATH_AFTER_MYDRIVE))

# Print current working directory
print("Current Working Directory:", os.getcwd())

# List files to make sure your data is visible
print("Available Files:", os.listdir())

If the output shows
> Mounted at /content/drive
> 
> Current Working Directory: /content/drive/My Drive/DeepLearning4ComputerVision/PA02
> 
> Available Files: ['datasets', 'dlcv', 'knn.py', 'softmax.py', ...]

Then, the Google Colab is working correctly.

### python env and ipynb setup (Local Machine Run from here) (Google Colab run this)

In [ ]:
import sys
import importlib
sys.modules["imp"] = importlib
# Notebook will reload external python modules;
%load_ext autoreload
%autoreload 2

import numpy as np

import matplotlib.pyplot as plt
# This is a bit of magic to make matplotlib figures appear inline in the notebook
# rather than in a new window.
%matplotlib inline
plt.rcParams['figure.figsize'] = (10.0, 8.0) # set default size of plots
plt.rcParams['image.interpolation'] = 'nearest'
plt.rcParams['image.cmap'] = 'gray'


def rel_error(x, y):
  """ returns relative error """
  return np.max(np.abs(x - y) / (np.maximum(1e-8, np.abs(x) + np.abs(y))))

## CIFAR-10 Data Loading and Preprocessing

In [ ]:
from dlcv.data_utils import get_CIFAR10_data
# Load the (preprocessed) CIFAR10 data.
cifar10_dir = 'datasets/cifar-10-batches-py'

data = get_CIFAR10_data(cifar10_dir)
num_train = 1000
small_data = {
  'X_train': data['X_train'][:num_train],
  'y_train': data['y_train'][:num_train],
  'X_val': data['X_val'],
  'y_val': data['y_val'],
}
for k, v in list(small_data.items()):
  print(('%s: ' % k, v.shape))

## Dropout

### Implement dropout forward pass

In `layers.py`, implement **dropout_forward()** function.

In [ ]:
from layers import dropout_forward
np.random.seed(231)
x = np.random.randn(500, 500) + 10

for p in [0.25, 0.4, 0.7]:
    out, _ = dropout_forward(x, {'mode': 'train', 'p': p})
    out_test, _ = dropout_forward(x, {'mode': 'test', 'p': p})

    print('Running tests with p = ', p)
    print('Mean of input: ', x.mean())
    print('Mean of train-time output: ', out.mean())
    print('Mean of test-time output: ', out_test.mean())
    print('Fraction of train-time output set to zero: ', (out == 0).mean())
    print('Fraction of test-time output set to zero: ', (out_test == 0).mean())
    print()

### Implement dropout backward pass

In `layers.py`, implement **dropout_backward()** function.

In [ ]:
from dlcv.gradient_check import eval_numerical_gradient_array
from layers import dropout_backward
np.random.seed(231)
x = np.random.randn(10, 10) + 10
dout = np.random.randn(*x.shape)

dropout_param = {'mode': 'train', 'p': 0.2, 'seed': 123}
out, cache = dropout_forward(x, dropout_param)
dx = dropout_backward(dout, cache)
dx_num = eval_numerical_gradient_array(lambda xx: dropout_forward(xx, dropout_param)[0], x, dout)

# Error should be around e-10 or less.
print('dx relative error: ', rel_error(dx, dx_num))

### Implement Two-layer-Net, apply dropout.

In `two_layer_net.py`, implement **`__init__()`** function in `class TwoLayerNet` with `'dropout'`.

In `two_layer_net.py`, implement **loss()** function in `class TwoLayerNet` with `'dropout'`.

For the Dropout Two-Layer-Net model:

$$
\boxed{
X
\rightarrow
\text{Linear}_1
\rightarrow
\text{BatchNorm}
\rightarrow
\text{ReLU}
\rightarrow
\text{Dropout}
\rightarrow
\text{Linear}_2
\rightarrow
\text{Softmax}
\rightarrow
\text{Cross-Entropy}.
}
$$

Notice: **Batch Normalization** is used

In [ ]:
from two_layer_net import TwoLayerNet
from dlcv.gradient_check import eval_numerical_gradient
np.random.seed(231)
N, input_dim, hidden_size, num_classes = 2, 15, 20, 10
X = np.random.randn(N, input_dim)
y = np.random.randint(num_classes, size=(N,))

# You should expect losses between 1e-4~1e-10 for W, 
# losses between 1e-08~1e-10 for b,
# and losses between 1e-08~1e-09 for beta and gammas.
model = TwoLayerNet(input_dim, hidden_size, num_classes, reg=0.01, normalization='batchnorm')

for dropout_keep_ratio in [1, 0.75, 0.5]:
    print('Running check with dropout = ', dropout_keep_ratio)
    
    model = TwoLayerNet(
        input_dim,
        hidden_size,
        num_classes,
        reg=0.01,
        normalization='batchnorm',
        dropout_keep_ratio=dropout_keep_ratio,
        seed=123
    )

    loss, grads = model.loss(X, y)
    print('Initial loss: ', loss)

    # Relative errors should be around e-6 or less.
    # Note that it's fine if for dropout_keep_ratio=1 you have W2 error be on the order of e-5.
    for name in sorted(grads):
        f = lambda _: model.loss(X, y)[0]
        grad_num = eval_numerical_gradient(f, model.params[name], verbose=False, h=1e-5)
        print('%s relative error: %.2e' % (name, rel_error(grad_num, grads[name])))
    print()

### Train witn Cifar10 data

In [ ]:
from dlcv.solver import Solver
np.random.seed(231)

input_size = 32 * 32 * 3
hidden_size = 200
num_classes = 10

solvers = {}
dropout_choices = [1, 0.5]
for dropout_keep_ratio in dropout_choices:
    model = model = TwoLayerNet(
        input_size,
        hidden_size,
        num_classes,
        reg=0.01,
        normalization='batchnorm',
        dropout_keep_ratio=dropout_keep_ratio)
    print(dropout_keep_ratio)

    solver = Solver(
        model,
        small_data,
        num_epochs=25,
        batch_size=100,
        update_rule='adam',
        optim_config={'learning_rate': 5e-4,},
        verbose=True,
        print_every=100
    )
    solver.train()
    solvers[dropout_keep_ratio] = solver
    print()

In [ ]:
# Plot train and validation accuracies of the two models.
train_accs = []
val_accs = []
for dropout_keep_ratio in dropout_choices:
    solver = solvers[dropout_keep_ratio]
    train_accs.append(solver.train_acc_history[-1])
    val_accs.append(solver.val_acc_history[-1])

plt.subplot(3, 1, 1)
for dropout_keep_ratio in dropout_choices:
    plt.plot(
        solvers[dropout_keep_ratio].train_acc_history, '-o', label='%.2f dropout_keep_ratio' % dropout_keep_ratio)
plt.title('Train accuracy')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend(ncol=2, loc='lower right')

plt.subplot(3, 1, 2)
for dropout_keep_ratio in dropout_choices:
    plt.plot(
        solvers[dropout_keep_ratio].val_acc_history, '-o', label='%.2f dropout_keep_ratio' % dropout_keep_ratio)
plt.title('Val accuracy')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend(ncol=2, loc='lower right')

plt.gcf().set_size_inches(15, 15)
plt.show()